# Test4Py Council — Git clone + engineering screen v4
Upload this notebook to Kaggle, enable **Internet** and **GPU T4 x2**, then Run All.
No source ZIP or API key is required for these public model/repository downloads.

This notebook clones a pinned base commit and applies an **embedded, SHA-256-checked
source overlay** containing the local runner fixes, helper and regression tests.
The overlay has not been pushed to GitHub. Both base commit and overlay hashes are
saved; this is a new development cohort, not an unchanged upstream checkout.

Fixed plan: 2 targets × 4 arms × 1 attempt = **8 planned outcomes**. No pruning,
no automatic full run, no claim of improvement over the paper. Failed outcomes
and raw evidence remain in the output ZIP. Runtime depends on model download and
inference; no fixed completion time is promised.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, subprocess, sys, zipfile

REPO_URL = 'https://github.com/nguyendevwk/Test4Py-Setup-pilot-temp.git'
BASE_COMMIT = '7a97eb0a6c8fefc2e8154d4c0b4487b4ae2b0ea3'
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORKDIR = Path('/kaggle/working') / ('Test4Py_screen_' + STAMP)
PREP = Path('/kaggle/working') / ('SCREEN_PREFLIGHT_' + STAMP)
OUTPUT = Path('/kaggle/working') / ('SCREEN_V4_' + STAMP)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
actual = subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual != BASE_COMMIT:
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', '--depth', '1', 'origin', BASE_COMMIT], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', 'HEAD'], text=True).strip() == BASE_COMMIT

# Transparent source snapshot, not model outputs or research measurements.
OVERLAY_SHA256 = {'council_multitarget.py': '5e738d7156ca2b43d0c810bfb659eb11b285d28724e9ff55a670ca961f8cd80f', 'council/generator.py': '1ecd5957fa02f3db2b7992e132f354da9fe1d2bc0e652bafda10c448cc8ab4c8', 'pilot/multitarget.py': '9f061ddaf7397a647a29a47cef30331e5aa1e8f23f6d307856619bf9085500bc', 'pilot/grounding.py': 'fe9be1949741591bcaffaacd337d75a4254ed1046f064b17f408c6463b624fa8', 'pilot/api_contract.py': '775fa5025f40852da9edd28551b377b4e28fd585e458fb9819f7ba291f606519', 'pilot/source_prompt.py': 'aa19865348f30a37f557fd7c1cbf2f1dfe548269baa886460dda1e3b8c4cb93f', 'tests/test_multitarget.py': '3df432a208590398d39f3eb3b7f2305d35c13b375169ed65448ce79b2e22341c', 'notebooks/kaggle_multitarget_smoke_v4.ipynb': '788e0f93051164c82c339218b064550ae3cc845a751fb3a6a4e146601157dafe'}
OVERLAY_ZIP_B64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(OVERLAY_ZIP_B64))) as archive:
    assert set(archive.namelist()) == set(OVERLAY_SHA256)
    for name, expected in OVERLAY_SHA256.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('Overlay integrity error: ' + name)
        destination = WORKDIR / name
        if not destination.resolve().is_relative_to(WORKDIR.resolve()):
            raise ValueError('Unsafe overlay path')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
provenance = dict(repository=REPO_URL, base_commit=BASE_COMMIT,
                  source_kind='pinned Git base plus embedded local v4 overlay',
                  overlay_sha256=OVERLAY_SHA256)
(WORKDIR/'notebook-source.json').write_text(json.dumps(provenance, indent=2))
os.chdir(WORKDIR)
print(json.dumps(provenance, indent=2))
print('Source:', WORKDIR, 'Output:', OUTPUT)


In [ ]:
# Prepare dependencies BEFORE the experiment; keep Kaggle CUDA Torch.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.6.17'], check=True)
subprocess.run([sys.executable, 'bootstrap_modern.py'], cwd=WORKDIR, check=True)
PYTHON = '/tmp/test4py-modern/bin/python'
subprocess.run([PYTHON, '-m', 'pip', 'install', 'bitsandbytes>=0.43,<1', 'accelerate>=0.29,<2'], check=True)
subprocess.run([PYTHON, '-c', "import torch; print(torch.__version__); print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]); assert torch.cuda.device_count() == 2"], check=True)


In [ ]:
# Real subprocess regression tests: no network/model inference in these tests.
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_multitarget.py', '-v'], cwd=WORKDIR, check=True)
subprocess.run([PYTHON, 'council_multitarget.py', '--prepare-only', '--output', str(PREP)], cwd=WORKDIR, check=True)
preflight = json.loads((PREP/'preflight.json').read_text())
for name, row in preflight.items():
    print(name, 'ready=', row['ready'], 'commit=', row.get('commit'), 'error=', row.get('error'))
assert all(row['ready'] for row in preflight.values())


The next cell runs the fixed eight-attempt GPU screen. Preparation has no model
inference. Proceed with the fixed plan; never choose targets or seeds based on
scores. Preserve this output even if readiness fails.


In [ ]:
cmd = [PYTHON, 'council_multitarget.py', '--prepared', str(PREP/'preflight.json'),
       '--targets', 'docstring_parser_parse,codetiming_timer', '--arms', 'baseline,grounded,ranked,verified',
       '--coder-revision', 'c03e6d358207e414f1eca0bb1891e29f1db0e242',
       '--critic-revision', 'cdbee75f17c01a7cc42f958dc650907174af0554',
       '--max-input-tokens', '8192', '--samples', '1', '--seed', '42', '--temperature', '0.7', '--max-new-tokens', '2048',
       '--coder-model', 'Qwen/Qwen2.5-Coder-7B-Instruct',
       '--critic-model', 'Qwen/Qwen3-4B-Instruct-2507', '--output', str(OUTPUT)]
# Tee bounded progress to notebook and preserve the complete launcher log.
launcher_log = Path('/kaggle/working') / ('SCREEN_LAUNCHER_' + STAMP + '.log')
with launcher_log.open('w') as stream:
    proc = subprocess.Popen(cmd, cwd=WORKDIR, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            stream.write(line)
            stream.flush()
            print(line, end='', flush=True)
        returncode = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        raise
run = subprocess.CompletedProcess(cmd, returncode)
if OUTPUT.exists():
    import shutil
    shutil.copy2(WORKDIR/'notebook-source.json', OUTPUT/'notebook-source.json')
    shutil.copy2(launcher_log, OUTPUT/'launcher.log')
    shutil.make_archive(str(OUTPUT), 'zip', OUTPUT)
print('Runner exit:', run.returncode, '(2 means readiness failed; inspect retained artifacts)')


In [ ]:
from IPython.display import display, Markdown, FileLink
if (OUTPUT/'FINAL_REPORT.md').exists():
    display(Markdown((OUTPUT/'FINAL_REPORT.md').read_text()))
if (OUTPUT/'readiness.json').exists():
    gate = json.loads((OUTPUT/'readiness.json').read_text())
    print(json.dumps(gate, indent=2))
archive_path = Path(str(OUTPUT)+'.zip')
if archive_path.exists():
    display(FileLink(str(archive_path)))
assert run.returncode == 0, 'Engineering gate failed. Review archive; do not start full experiments.'
print('Automated gate passed. Assertion/target-reach review is still required. No full run launched.')
